In [ ]:
import requests
from bs4 import BeautifulSoup
import os
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
# response = requests.get('https://www.merit-times.com.tw/NewsPage.aspx?unid=903451')
response = requests.get('https://www.merit-times.com.tw/NewsPage.aspx?unid=903292')
soup = BeautifulSoup(response.text,"html.parser")
body_content = soup.body
body_content_str = str(body_content)
soup1 = BeautifulSoup(body_content_str,'html.parser')
for script_or_style in soup(['script','style']):
    script_or_style.extract()

cleaned_content = soup.get_text(separator="\n")
cleaned_content = "\n".join(
    line.strip() for line in cleaned_content.splitlines() if line.strip()
)
#print(cleaned_content)
#print(len([cleaned_content[i : i + 6000] for i in range(0, len(cleaned_content), 6000)]))

system_instruction = '''
    你的任務是取出文字的內容:
    規則:
    1. 當看到`文／作者名稱`,作者名稱可以是任何名稱,下面的文字就是我要的文字
    2. 當看到`前一篇文章`的文字時,上面就是我要的內容
    3. 請整理文字內容,成為適合閱讀的文字(不修改裏面的內容)
    '''

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=cleaned_content,
    config=types.GenerateContentConfig(system_instruction=system_instruction)
)
print(response.text)